In this notebook, we will replace the growth kernel in Trial IPM.ipynb with a Gaussian Process. 

In [18]:
import pandas as pd
import numpy as np
import jax.numpy as jnp
import numpyro
import matplotlib.pyplot as plt
import jax
import numpyro.distributions as dist
import arviz as az
from jax.random import PRNGKey

from numpyro.infer import (
    MCMC,
    NUTS,
    Predictive,
    SVI,
    Trace_ELBO,
)

from numpyro.infer.autoguide import AutoLowRankMultivariateNormal
from numpyro.optim import Adam

In [2]:
sim_data = pd.read_csv("sim_data.csv")

In [3]:
growth_data = sim_data.dropna(subset = ["z1"]).copy() # Keep only observations which have survived (and hence could have grown)
X = growth_data["z"].values
y = growth_data["z1"].values

In [4]:
X_mean = X.mean()
X_std = X.std()
X_scaled = (X - X_mean)/X_std
y_mean = y.mean()
y_std = y.std()
y_scaled = (y - y_mean) / y_std


In [5]:
def rbf(X1, X2, alpha, ell):
    """
    Squared Exponential (RBF) kernel.

    Parameters
    ----------
    X1, X2 : jax array of shape (n, ), (m, )
        Input locations.

    alpha : float
        Output scale.

    ell : float
        Length scale.

    Returns
    -------
    K : jax array of shape (n, n)
        Covariance matrix.
    """

    # Pairwise squared distances
    sq_dist = (X1[:, None] - X2[None, :]) ** 2

    # Covariance matrix
    K = alpha**2 * jnp.exp(-0.5 * sq_dist / ell**2)

    return K

In [6]:
def hyperparameter_prior(X):
    """
    Construct the GP prior.

    Returns
    -------
    alpha : output scale
    ell : length scale
    K : covariance matrix
    """

    alpha = numpyro.sample(
        "alpha",
        dist.HalfNormal(5.0)
    )

    ell = numpyro.sample(
        "ell",
        dist.HalfNormal(5.0)
    )

    K = rbf(
        X,
        X,
        alpha,
        ell
    )

    K = K + 1e-6 * jnp.eye(len(X))

    return alpha, ell, K

In [10]:
def gp_growth_model(X, y = None):
    _, _, K = hyperparameter_prior(X)

    sigma = numpyro.sample(
        "sigma",
        dist.HalfNormal(100.0)
    )
    K_y = K + (sigma**2 + 1e-6) * jnp.eye(len(X))

    numpyro.sample(
        "y",
        dist.MultivariateNormal(
            loc = jnp.zeros(len(X)),
            covariance_matrix = K_y
        ),
        obs =y
        )




In [11]:
def fit_gp_model(
    model,
    X,
    y,
    num_warmup=1000,
    num_samples=2000,
    num_chains=1,
    rng_key=PRNGKey(0),
):
    """
    Fit a GP model using NUTS.

    Parameters
    ----------
    model : callable
        NumPyro model.

    X : jax array
        Input locations.

    y : jax array
        Observations.

    Returns
    -------
    mcmc : fitted MCMC object
    """

    kernel = NUTS(model)

    mcmc = MCMC(
        sampler=kernel,
        num_warmup=num_warmup,
        num_samples=num_samples,
        num_chains=num_chains,
        progress_bar=True,
    )

    mcmc.run(
        rng_key,
        X=X,
        y=y,
    )

    return mcmc

In [ ]:
growth_mcmc = fit_gp_model(
    gp_growth_model,
    X_scaled,
    y_scaled
)

growth_posterior = growth_mcmc.get_samples()

sample: 100%|██████████| 3000/3000 [00:39<00:00, 76.78it/s, 7 steps of size 5.09e-01. acc. prob=0.88]


NameError: name 'growth' is not defined

In [ ]:
def gp_posterior(
    X_train, 
    y_train,
    X_pred,
    alpha,
    ell,
    sigma
):
"""
    Posterior predictive distribution of the latent GP.

    Parameters
    ----------
    X_train : (n,)
    y_train : (n,)
    X_pred  : (m,)
    alpha   : GP output scale
    ell     : GP length scale
    sigma   : Observation noise

    Returns
    -------
    posterior_mean : (m,)
    posterior_cov  : (m, m)
    """
    K = rbf(X_train, X_train, alpha, ell)
    K_star = rbf(X_train, X_pred, alpha, ell)
    K_starstar = rbf(X_pred, X_pred, alpha, ell)

    K_y = K + (sigma**2 + 1e-6) * jnp.eye(len(X_train))

    v = jax.scipy.linalg.solve_triangular(
        L,
        y_train,
        lower = True
    )

    alpha_vec = jax.scipy.linalg.solve_triangular(
        L.T,
        v,
        lower=False
    )

    posterior mean = K_star.T @ alpha_vec

    v = jax.scipy.linalg.solve_triangular (
        L, 
        K_Star,
        lower = True
    )

    posterior_cov = K_starstar - v.T @ v
    return posterior_mean, posterior_cov


In [29]:
def gp_bernoulli_model(X, y = None):
    """
    Bayesian GP classification model.

    Parameters
    ----------
    X : input locations
    y : binary observations (0 or 1)
    """

    alpha, ell, K = hyperparameter_prior(X)

    # Latent GP
    f = numpyro.sample(
        "f",
        dist.MultivariateNormal(
            loc=jnp.zeros(len(X)),
            covariance_matrix=K,
        ),
    )

    # Bernoulli likelihood with logistic link
    numpyro.sample(
        "y",
        dist.Bernoulli(logits=f),
        obs=y,
    )

In [14]:
survival_data = sim_data[
    sim_data["Repr"] == 0
].copy()

X_surv = survival_data["z"].to_numpy()
y_surv = survival_data["Surv"].to_numpy()

In [15]:
X_surv_mean = X_surv.mean()
X_surv_std = X_surv.std()

X_surv_scaled = (
    X_surv - X_surv_mean
) / X_surv_std

In [16]:
X_surv.shape

(961,)

In [56]:
X_surv_scaled = jnp.asarray(X_surv_scaled)
y_surv = jnp.asarray(y_surv)

In [ ]:
def fit_gp_vi(
    model,
    X,
    y,
    learning_rate = 1e-3,
    rank = 20,
    num_steps = 10_000,
    rng_key = PRNGKey(0)
):
    """
    Performs Variational Infernce over the posterior.
    Recall, for VI, we specify a variational distribution parameterized
    by lambda, and find lambda that minimizes the KL divergence between
    the posterior and the variational distribution. 
    This turns out to be equivalent to maximizing the ELBO 
    """
    guide = AutoLowRankMultivariateNormal(model, rank = rank)
    optimizer = Adam(learning_rate)
    svi = SVI(
        model = model,
        guide = guide,
        optim = optimizer,
        loss = Trace_ELBO()
    )
    
    svi_result = svi.run(
        rng_key,
        num_steps,
        X=X,
        y=y
    )

    return guide, svi_result

In [46]:
survival_guide, survival_result = fit_gp_vi(
    gp_bernoulli_model,
    X_surv_scaled,
    y_surv,
)

100%|██████████| 10000/10000 [04:08<00:00, 40.17it/s, init loss: 655915072.0000, avg. loss [9501-10000]: nan]


In [31]:
predictive = Predictive(
    survival_guide,
    params=survival_result.params,
    num_samples=2000,
)

survival_posterior = predictive(
    PRNGKey(1),
    X=X_surv_scaled,
    y=None,
)